4.5.2. Download the implementation from https://github.com/kkirchheim/odd-coverage and compute the k-projection coverage of your test set for k ∈ {1,2,3}.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Add odd-coverage folder to Python path
sys.path.append(str(PROJECT_ROOT / "odd-coverage"))

from kprojection import KProjectionCoverage

# Use your already extracted test data from earlier exercises
TEST_DIR = PROJECT_ROOT / "data" / "exercise_3_6" / "test"

test_csv = sorted(TEST_DIR.rglob("*.csv"))[0]
test_df = pd.read_csv(test_csv)

print("Test shape:", test_df.shape)
print(test_df.columns.tolist())
display(test_df.head())

Test shape: (3600, 7)
['frame', 'has_traffic_light', 'has_pedestrian', 'has_vehicle', 'px_traffic_light', 'px_pedestrian', 'px_vehicle']


,frame,has_traffic_light,has_pedestrian,has_vehicle,px_traffic_light,px_pedestrian,px_vehicle
0,0,False,False,False,15,0,35
1,10,True,False,True,299,0,116
2,20,True,False,True,298,0,307
3,30,True,False,True,297,0,258
4,40,True,False,True,297,0,249


In [2]:
#define ODD dimensions - the test only contains daytime CARLA data
description = {
    "weather": ["clear"],
    "lighting": ["daytime"],
    "camera": ["forward_fixed"],
    "scene": ["urban"],
    "traffic_light": [False, True],
    "pedestrian": [False, True],
    "vehicle": [False, True],
}

#converting each test image into a scenario
scenarios = []

for _, row in test_df.iterrows():
    scenario = {
        "weather": "clear",
        "lighting": "daytime",
        "camera": "forward_fixed",
        "scene": "urban",
        "traffic_light": bool(row["has_traffic_light"]),
        "pedestrian": bool(row["has_pedestrian"]),
        "vehicle": bool(row["has_vehicle"]),
    }
    scenarios.append(scenario)

print("Number of scenarios:", len(scenarios))
print(scenarios[0])

Number of scenarios: 3600
{'weather': 'clear', 'lighting': 'daytime', 'camera': 'forward_fixed', 'scene': 'urban', 'traffic_light': False, 'pedestrian': False, 'vehicle': False}


In [3]:
#k-projection coverage for k = 1, 2, 3
results = []

for k in [1, 2, 3]:
    cov = KProjectionCoverage(k=k, desc=description)
    cov.add_scenarios(scenarios)
    result = cov.compute()

    results.append({
        "k": result.k,
        "covered": result.covered,
        "total": result.total,
        "coverage": result.coverage,
        "coverage_percent": round(result.coverage * 100, 2),
        "scenarios": result.scenes
    })

coverage_df = pd.DataFrame(results)
display(coverage_df)

,k,covered,total,coverage,coverage_percent,scenarios
0,1,10,10,1.0,100.0,3600
1,2,42,42,1.0,100.0,3600
2,3,96,96,1.0,100.0,3600


In [4]:
#save results
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

coverage_df.to_csv(RESULTS_DIR / "exercise_4_5_k_projection_coverage.csv", index=False)

print("Saved coverage result.")

Saved coverage result.
